In [0]:
from pyspark.sql.functions import col, year, month, to_date

# 1. Load Databricks' built-in sample table directly
df = spark.read.table("samples.nyctaxi.trips")

# 2. Inspect the columns and clean up basic fields
# The built-in columns are: tpep_pickup_datetime, trip_distance, fare_amount, etc.
df_clean = df.filter(
    (col("trip_distance") > 0) & 
    (col("fare_amount") > 0)
).withColumn("pickup_date", to_date(col("tpep_pickup_datetime"))) \
 .withColumn("pickup_year", year(col("pickup_date"))) \
 .withColumn("pickup_month", month(col("pickup_date")))

# 3. Spark SQL: Register as a view and run an aggregation
df_clean.createOrReplaceTempView("trips")

sql_query = """
    SELECT 
        pickup_year,
        pickup_month,
        COUNT(*) AS total_trips,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(SUM(fare_amount), 2) AS total_revenue
    FROM trips
    GROUP BY pickup_year, pickup_month
    ORDER BY pickup_year DESC, pickup_month DESC
"""
agg_results = spark.sql(sql_query)
display(agg_results)

# 4. Save as Partitioned Parquet (Satisfies M1 Milestone Requirement)
output_path = "/tmp/m1_nyc_taxi_partitioned"

df_clean.write \
    .mode("overwrite") \
    .partitionBy("pickup_year", "pickup_month") \
    .saveAsTable("m1_nyc_taxi_partitioned")

print("SUCCESS: Notebook 1 complete and partitioned table written!")

pickup_year,pickup_month,total_trips,avg_distance,total_revenue
2016,2,11018,2.84,135845.02
2016,1,10829,2.88,133177.01


SUCCESS: Notebook 1 complete and partitioned table written!


In [0]:
import time
from pyspark.sql.functions import col

# 1. Create an unpartitioned table to compare against
df_clean.write \
    .mode("overwrite") \
    .saveAsTable("m1_nyc_taxi_unpartitioned")

# Test parameters (matching values from your data output)
target_year = 2016
target_month = 1

# Benchmark 1: Querying the Unpartitioned Table (Full scan)
t0 = time.time()
df_unpart = spark.read.table("m1_nyc_taxi_unpartitioned")
count_unpart = df_unpart.filter(
    (col("pickup_year") == target_year) & (col("pickup_month") == target_month)
).count()
time_unpart = time.time() - t0
print(f"Unpartitioned query time : {time_unpart:.3f} s (Rows: {count_unpart})")

# Benchmark 2: Querying the Partitioned Table (Partition pruning active)
t1 = time.time()
df_part = spark.read.table("m1_nyc_taxi_partitioned")
count_part = df_part.filter(
    (col("pickup_year") == target_year) & (col("pickup_month") == target_month)
).count()
time_part = time.time() - t1
print(f"Partitioned query time   : {time_part:.3f} s (Rows: {count_part})")

# 2. Execution plan evidence (Proves partition filter to your mentor)
print("\n--- Physical Execution Plan (Showing PartitionFilters) ---")
df_part.filter(
    (col("pickup_year") == target_year) & (col("pickup_month") == target_month)
).explain(True)

Unpartitioned query time : 0.831 s (Rows: 10829)
Partitioned query time   : 0.626 s (Rows: 10829)

--- Physical Execution Plan (Showing PartitionFilters) ---
== Parsed Logical Plan ==
'Filter 'and('`==`('pickup_year, 2016), '`==`('pickup_month, 1))
+- 'UnresolvedRelation [m1_nyc_taxi_partitioned], [], false

== Analyzed Logical Plan ==
tpep_pickup_datetime: timestamp, tpep_dropoff_datetime: timestamp, trip_distance: double, fare_amount: double, pickup_zip: int, dropoff_zip: int, pickup_date: date, pickup_year: int, pickup_month: int
Filter ((pickup_year#11508 = 2016) AND (pickup_month#11509 = 1))
+- SubqueryAlias workspace.default.m1_nyc_taxi_partitioned
   +- Relation workspace.default.m1_nyc_taxi_partitioned[tpep_pickup_datetime#11501,tpep_dropoff_datetime#11502,trip_distance#11503,fare_amount#11504,pickup_zip#11505,dropoff_zip#11506,pickup_date#11507,pickup_year#11508,pickup_month#11509] parquet

== Optimized Logical Plan ==
Filter ((isnotnull(pickup_year#11508) AND isnotnull(pickup

In [0]:
from pyspark.sql.functions import col, window

# 1. Ensure a working Volume exists in your default catalog/schema
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.m1_checkpoints")

# 2. Synthetic rate stream (5 events per second)
rate_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# 3. 10-second tumbling window aggregation
windowed_rate = rate_stream.groupBy(
    window(col("timestamp"), "10 seconds")
).count()

# 4. Display with explicit Volume checkpoint
display(
    windowed_rate,
    checkpointLocation="/Volumes/workspace/default/m1_checkpoints/rate_demo"
)

Checkpointing to /Volumes/workspace/default/m1_checkpoints/rate_demo
